Exercice de dev pour l'entrainement d'un modèle
17/11/26

Étape 1 : Exploration des Données et Diagnostic Qualité

À l'aide de Pandas, 
- inspectez les types de variables = voir si catégrorielle (ou binaire) ou numérique -> df.dtypes
- quantifiez les valeurs manquantes et autres
  * vérifier que les valeurs sont non nulles df.info(): affiche le nombre de valeur non nulles
  * vérifier les valeurs aberrantes (il ne devrait pas y en avoir dans ce cas de test); 
  faire df.describe(include="all") pour avoir les min max etc.
  faire des tests sur les valeurs (vérifier que l'age et inf à 18 ou sup à X etc)
  * vérifier les valeurs manquantes
- analysez la distribution de la cible decision_octroi pour identifier l'éventuel déséquilibre de classes. 
- Veillez à ne jamais supprimer arbitrairement des lignes sans justification documentée.


Notes sur les colonnes 

id_dossier (DOS-10000)
montant_demande (16100.0)
duree_mois (48)
revenu_mensuel (3800.0)
taux_endettement (0.088)
age (27)
sexe (M)
statut_marital (celibataire)
nationalite (FR)
situation_logement (proprietaire)
anciennete_emploi_mois (31)
type_contrat (CDI)
epargne_globale (1820.0)
nb_credits_actifs (0)
incident_precedent (retard_leger)
score_credit (621.0)
decision_octroi (1)

In [17]:
###################################################
# Afficher les informations sur le dataset
# Inspectez les types de variables
################################################### 

from pathlib import Path

import pandas as pd

# Chargement direct et robuste du dataset synthétique
candidates = [
    Path('credit_risk_dataset.csv')
]
csv_path = next((p for p in candidates if p.exists()), Path('credit_risk_dataset.csv'))
df = pd.read_csv(csv_path)

print(f'Données chargées depuis : {csv_path.resolve()}')
print('Dimensions du dataset :', df.shape)

# Aperçu des premières lignes du dataset (5 par défaut)
print('---------------------- head du dataset (head) ----------------------')
df.head()


Données chargées depuis : C:\Workspace\Simplon\SocleApplicatif-module0\dev\credit_risk_dataset.csv
Dimensions du dataset : (1200, 17)
---------------------- head du dataset (head) ----------------------


,id_dossier,montant_demande,duree_mois,revenu_mensuel,taux_endettement,age,sexe,statut_marital,nationalite,situation_logement,anciennete_emploi_mois,type_contrat,epargne_globale,nb_credits_actifs,incident_precedent,score_credit,decision_octroi
0,DOS-10000,16100.0,48,3800.0,0.088,27,M,celibataire,FR,proprietaire,31,CDI,1820.0,0,retard_leger,621.0,1
1,DOS-10001,11100.0,12,3030.0,0.305,20,F,marie,FR,proprietaire,281,CDI,5930.0,0,aucun,693.8,1
2,DOS-10002,16100.0,12,3980.0,0.337,19,M,divorce,FR,locataire,38,CDI,980.0,2,aucun,677.8,1
3,DOS-10003,19900.0,72,5030.0,0.055,24,F,divorce,HORS_UE,proprietaire,48,CDD,20490.0,0,retard_leger,634.4,1
4,DOS-10004,15400.0,24,2920.0,0.220,32,F,divorce,FR,locataire,45,CDI,3570.0,1,retard_leger,634.2,1


In [19]:
# Informations détaillées sur le dataset
print('---------------------- informations détaillées (info) ----------------------')
# Indique les types et les non null par colonne
df.info()        

# Types de variables par colonne
print('---------------------- types de variables (dtypes) ----------------------')
print('Types de variables par colonne :', df.dtypes)

---------------------- informations détaillées (info) ----------------------
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id_dossier              1200 non-null   str    
 1   montant_demande         1200 non-null   float64
 2   duree_mois              1200 non-null   int64  
 3   revenu_mensuel          1200 non-null   float64
 4   taux_endettement        1200 non-null   float64
 5   age                     1200 non-null   int64  
 6   sexe                    1200 non-null   str    
 7   statut_marital          1200 non-null   str    
 8   nationalite             1200 non-null   str    
 9   situation_logement      1200 non-null   str    
 10  anciennete_emploi_mois  1200 non-null   int64  
 11  type_contrat            1200 non-null   str    
 12  epargne_globale         1200 non-null   float64
 13  nb_credits_actifs       120

In [28]:
# Statistiques descriptives pour toutes les colonnes
# Résumé statistique de chaque variable numérique
# Permet de repérer des valeurs suspectes sans regarder les 1 200 lignes
# Affiche pour chaque colonne:
# - count: Nombre de valeurs non manquantes. Si c’est inférieur au nombre de lignes, il y a des manquants.
# - mean: moyenne
# - std: dispersion des valeurs autour de la moyenne
# - min / max: Valeurs extrêmes; c’est le plus utile pour détecter des erreurs.
# - 25% / 50% / 75%: Quartiles. La médiane (50 %) décrit le cas « typique »
# - unique: Nombre de valeurs uniques (utile pour les variables catégorielles)
# - top: Valeur la plus fréquente (utile pour les variables catégorielles)
# - freq: Fréquence de la valeur la plus fréquente (utile pour les variables catégorielles)
print('---------------------- statistiques descriptives (describe) ----------------------')
print(df.describe(include="all"))

# ages impossibles
print(f'Age inférieur à 18 : {(df["age"] < 18).sum()}')
print(f'Revenu mensuel inférieur ou égal à 0 : {(df["revenu_mensuel"] <= 0).sum()}')
print(f'Taux d\'endettement : {(df["taux_endettement"] > 1).sum()}')
print(f'Durée (mois) inférieure ou égale à 0 : {(df["duree_mois"] <= 0).sum()}')
print(f'Ancienneté emploi (mois) supérieur à l\'âge en mois : {(df["anciennete_emploi_mois"] > df["age"]*12).sum()}')

---------------------- statistiques descriptives (describe) ----------------------
       id_dossier  montant_demande   duree_mois  revenu_mensuel  \
count        1200      1200.000000  1200.000000     1200.000000   
unique       1200              NaN          NaN             NaN   
top     DOS-10000              NaN          NaN             NaN   
freq            1              NaN          NaN             NaN   
mean          NaN     15460.916667    40.510000     3264.175000   
std           NaN      8372.553207    17.985813     1147.409521   
min           NaN      1500.000000    12.000000     1200.000000   
25%           NaN      9200.000000    24.000000     2450.000000   
50%           NaN     15100.000000    36.000000     3260.000000   
75%           NaN     21100.000000    48.000000     4012.500000   
max           NaN     43700.000000    84.000000     7820.000000   

        taux_endettement          age  sexe statut_marital nationalite  \
count        1200.000000  1200.000000 

In [ ]:
# vérifier les valeurs manquantes
df.isna().sum()

id_dossier                0
montant_demande           0
duree_mois                0
revenu_mensuel            0
taux_endettement          0
age                       0
sexe                      0
statut_marital            0
nationalite               0
situation_logement        0
anciennete_emploi_mois    0
type_contrat              0
epargne_globale           0
nb_credits_actifs         0
incident_precedent        0
score_credit              0
decision_octroi           0
dtype: int64

In [33]:
# Analyser la cible decision_octroi
print('Valeurs de la cible decision_octroi :')
print(df["decision_octroi"].value_counts(dropna=False))
# ==> il y a déséquilibre de classe
# Cela aura des conséquences pour la suite : choix de métriques (F1, AUC-PR, rappel plutôt que l’accuracy), stratification du split, éventuel rééquilibrage (class_weight, SMOTE).

df["decision_octroi"].value_counts(normalize=True).round(3)

Valeurs de la cible decision_octroi :
decision_octroi
0    748
1    452
Name: count, dtype: int64


decision_octroi
0    0.623
1    0.377
Name: proportion, dtype: float64

Analyse:

Type relevés:

1) catégorielles (str)
sexe, statut_marital, nationalite, situation_logement, type_contrat, incident_precedent

2) numériques
montant_demande, duree_mois, revenu_mensuel, taux_endettement, age, anciennete_emploi_mois, epargne_globale, nb_credits_actifs

3) Décision de suppression de certaines valeurs avec justification
- id_dossier: identifiant (à ne pas utiliser comme variable).
- decision_octroi: à ne pas utiliser comme valeur, c'est l'output
- score_credit: à ne pas utiliser comme valeur, c'est l'output


Étape 2 : Feature Engineering Raisonné
Les modèles linéaires et arborescents gagnent considérablement en précision lorsque l'on explicite des ratios financiers clés. 
Vous êtes invités à concevoir au moins deux variables dérivées métiers, par exemple :
●	Ratio Reste à Vivre : Revenu mensuel diminué de la charge de crédit estimée.
●	Ratio Épargne / Emprunt : Épargne globale rapportée au montant total sollicité.


In [ ]:

# Mensualité du nouveau crédit (remboursement linéaire, sans intérêts)
df["mensualite_estimee"] = df["montant_demande"] / df["duree_mois"]

# Charge des crédits déjà en cours
df["charge_existante"] = df["taux_endettement"] * df["revenu_mensuel"]

# 1. Reste à vivre (en €)
df["reste_a_vivre"] = (df["revenu_mensuel"]
                       - df["charge_existante"]
                       - df["mensualite_estimee"])


# 2. Reste à vivre en proportion du revenu
df["ratio_reste_a_vivre"] = df["reste_a_vivre"] / df["revenu_mensuel"]

# 3. Ratio épargne / emprunt
df["ratio_epargne_emprunt"] = df["epargne_globale"] / df["montant_demande"]

# 4. Taux d'endettement après octroi
#df["taux_endettement_projete"] = ((df["charge_existante"] + df["mensualite_estimee"]) / df["revenu_mensuel"])

# 5. Montant demandé rapporté au revenu annuel
#df["montant_sur_revenu_annuel"] = df["montant_demande"] / (df["revenu_mensuel"] * 12)

# 6. Indicateur d'incohérence ancienneté / âge (1 = incohérent)
#df["anciennete_incoherente"] = (df["anciennete_emploi_mois"] > (df["age"] - 16) * 12).astype(int)

Étape 3 : Construction du Pipeline Scikit-Learn Étanche
Pour éviter toute fuite d'information (Data Leakage) entre le train et le test, l'intégralité du prétraitement doit être encapsulée dans un ColumnTransformer standardisé :
●	Branche numérique : Imputation par la médiane (SimpleImputer) suivie d'une standardisation (StandardScaler).
●	Branche catégorielle : Imputation par constante 'INCONNU' suivie d'un encodage OneHot (OneHotEncoder avec handle_unknown='ignore').
●	Pipeline unifié : Assemblage du ColumnTransformer et de l'estimateur dans un objet Pipeline unique.

In [ ]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_features = ["montant_demande", "duree_mois", "revenu_mensuel", "taux_endettement", "age", "anciennete_emploi_mois", "epargne_globale", "nb_credits_actifs"]
cat_features = ["sexe", "statut_marital", "nationalite", "situation_logement", "type_contrat", "incident_precedent"]

num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

#Créer une colonne « inconnu » pour le reste des données
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="INCONNU")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])


preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, num_features),
    ("cat", cat_transformer, cat_features)
])

# Pipeline complet : Prétraitement étanche + Modèle unifié
full_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

# Sauvegarder le pipeline complet pour une utilisation ultérieure
joblib.dump(full_pipeline, "full_pipeline.pkl")
print("Pipeline sauvegardé sous 'full_pipeline.pkl'")


Étape 4 : Validation Croisée K-Fold Stratifiée

Il est formellement interdit d'ajuster vos hyperparamètres sur le jeu de test final. 
Vous devez configurer une validation croisée à 5 plis stratifiés (StratifiedKFold) sur vos données d'entraînement. 
Vos scores de comparaison intermédiaires doivent obligatoirement être présentés sous forme moyenne +/- écart-type.


In [34]:
from sklearn.model_selection import StratifiedKFold

# Configuration des 5 plis stratifiés avec mélange aléatoire contrôlé
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
